# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassantahir-afk/ML-Engineering-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page (content_hash_id), for one client, on one specific day (report_date).

The Table i will use is fact_content_daily_performance because it gives most readily avaliable fresh recent content(giving the current state).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# DESCRIBE just reads schema/metadata, doesn't scan the actual data
schema = con.execute(f"""
    DESCRIBE SELECT client_hash_id, content_hash_id, report_date FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()
print(schema)

       column_name column_type null   key default extra
0   client_hash_id     VARCHAR  YES  None    None  None
1  content_hash_id     VARCHAR  YES  None    None  None
2      report_date        DATE  YES  None    None  None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features:
gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users,
ga4_engaged_sessions, ga4_total_engagement_sec, scroll_events

### Context:
report_date, content_hash_id, client_hash_id

Availability flags (used to filter, not as features): client_has_gsc, client_has_ga4,
gsc_data_available, ga4_data_available

### Label:
declining_flag: TRUE if a page's gsc_impressions dropped 10% or more from the first half of
the month to the second half.

first_half = sum of gsc_impressions from day 1 to day 15 of the month
second_half = sum of gsc_impressions from day 16 to end of month
pct_change = (second_half - first_half) / first_half * 100
declining_flag = TRUE if pct_change <= -10, else FALSE

Computed only on rows where gsc_data_available IS TRUE, and only for pages with nonzero
first_half impressions (a zero-impression starting point makes percent change undefined).

### Excluded:
gsc_sum_position (redundant with gsc_avg_position, which already captures the same signal.)

sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid,
sessions_ai (excluded because Lane 2 focuses on overall page decline, not traffic-source
attribution.)

ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other (excluded
because AI-referral data is extremely sparse (only 30,177 rows have any AI sessions, out of
78.8M total rows), too thin to be a reliable feature at this stage.)

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

result = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            CASE WHEN report_date < DATE '2026-03-16' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    agg AS (
        SELECT
            content_hash_id,
            client_hash_id,
            period,
            SUM(gsc_impressions) AS total_impressions
        FROM daily
        GROUP BY content_hash_id, client_hash_id, period
    ),
    pivoted AS (
        SELECT
            content_hash_id,
            client_hash_id,
            MAX(CASE WHEN period = 'first_half' THEN total_impressions ELSE 0 END) AS first_half,
            MAX(CASE WHEN period = 'second_half' THEN total_impressions ELSE 0 END) AS second_half
        FROM agg
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT *,
        (second_half - first_half) * 1.0 / NULLIF(first_half, 0) * 100 AS pct_change,
        CASE WHEN (second_half - first_half) * 1.0 / NULLIF(first_half, 0) * 100 <= -10 THEN TRUE ELSE FALSE END AS declining_flag
    FROM pivoted
    WHERE first_half > 0
""").df()

print(result.head())
print(f"\nTotal pages: {len(result)}")
print(f"Declining pages: {result['declining_flag'].sum()} ({result['declining_flag'].mean()*100:.1f}%)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id           client_hash_id  first_half  second_half  \
0  content_905aa32a0230694e  client_73cda7b4e4f265ea        89.0         60.0   
1  content_36c36abc7650d7af  client_73cda7b4e4f265ea      3705.0       1925.0   
2  content_05434271b257bb68  client_73cda7b4e4f265ea       628.0        793.0   
3  content_22610b0934f8825e  client_73cda7b4e4f265ea        41.0         26.0   
4  content_712c365258cee05c  client_73cda7b4e4f265ea      2531.0       3517.0   

   pct_change  declining_flag  
0  -32.584270            True  
1  -48.043185            True  
2   26.273885           False  
3  -36.585366            True  
4   38.956934           False  

Total pages: 151981
Declining pages: 58658 (38.6%)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 Grain:**  Zero Rows Returned, which signals that One Page = One Row per Day per Client and that there are no duplicate rows.

**Query 2 Row count + date span:** shows the number of pages avaliable within a time frame of a month. This slice contains 9,841,378 rows, spanning exactly
2026-03-01 to 2026-03-31

**Query 3 Availability:** Shows the number of pages out of those pages within the time frame of a month that i usable. Here it shows Only 3,611,061 rows (36.7%) have gsc_data_available = TRUE, and
only 413,966 rows (4.2%) have ga4_data_available = TRUE. This confirms the unbalanced-panel
warning from the data documentation most rows in this month lack trustworthy search or
analytics data. Any feature or label built from GSC or GA4 columns must filter on
these flags first, or risk treating zero-filled placeholder data as real signal.

### Three — five features, max:

[gsc_impressions] Knowable at the decision moment because it's a daily search-console
measurement.

[gsc_clicks] Same reasoning as gsc_impressions, a daily search-console measurement,
restricted to the first-half window only, so it reflects only what was already known
before the decision point.

[gsc_avg_position] Same reasoning, a daily search-console measurement of ranking position,
restricted to the first-half window, so it does not leak information from the period the
label measures.

Note: I chose only 3 features rather than the full 5 allowed, since GSC-based signals had
far higher availability in this slice (36.7%, per Query 3) than GA4-based signals (4.2%).
Rather than including a weaker GA4 feature just to reach 5, I kept the feature set limited
to the most reliable signals available.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.execute(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS row_count
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("=== Grain Check ===")
print(f"Rows violating the grain: {len(grain_check)}")
print(grain_check)
print("If the above returns zero rows then it will be confirmed that there are no duplicate values in the dataset\n One Row = One Page per Client per Day\n")

count_and_span = con.execute(f"""
    SELECT COUNT(*) AS total_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print("=== Count and Span ===")
print(count_and_span)


availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print("\n=== Availability Check ===")
print(availability_check)

# ─────────────────────────────────────────────
# The Construction of the 5-fearture dataframe
# ─────────────────────────────────────────────

feature_frame = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            CASE WHEN report_date < DATE '2026-03-16' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    features AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS feat_impressions,
            SUM(gsc_clicks) AS feat_clicks,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS feat_avg_position,
            COUNT(*) AS feat_days_active,
            SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS feat_ctr
        FROM daily
        WHERE period = 'first_half'
        GROUP BY content_hash_id, client_hash_id
    ),
    label AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN period = 'second_half' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.feat_impressions,
        f.feat_clicks,
        f.feat_avg_position,
        f.feat_days_active,
        f.feat_ctr,
        CASE
            WHEN (l.second_half - l.first_half) * 1.0 / NULLIF(l.first_half, 0) * 100 <= -10
            THEN TRUE ELSE FALSE
        END AS declining_flag
    FROM features f
    JOIN label l
        ON f.content_hash_id = l.content_hash_id AND f.client_hash_id = l.client_hash_id
    WHERE l.first_half > 0
""").df()

print("\n=== Feature Frame ===")
print(feature_frame.shape)
print(feature_frame.head())

from sklearn.tree import DecisionTreeClassifier, export_text
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# ─────────────────────────────────────────────
# 1) FIVE FEATURE MODEL
# ─────────────────────────────────────────────

features = ["feat_impressions", "feat_clicks", "feat_avg_position", "feat_days_active", "feat_ctr"]

y = feature_frame['declining_flag']
X = feature_frame[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

tree_score = tree.predict_proba(X)[:, 1]

print("\n=== SELECTED FEATURE MODEL ===")
print(f"Precision@50: {precision_at_k(tree_score, y, 50)}")
print(export_text(tree, feature_names=features))

# ─────────────────────────────────────────────
# 2) LEAKY FEATURE MODEL
# ─────────────────────────────────────────────

leaky_features = ["feat_impressions", "feat_clicks", "feat_avg_position", "feat_days_active", "feat_ctr", "declining_flag"]

X_leaky = feature_frame[leaky_features].replace([np.inf, -np.inf], np.nan).fillna(0)

leaky_tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
leaky_tree.fit(X_leaky, y)

leaky_tree_score = leaky_tree.predict_proba(X_leaky)[:, 1]

print("\n=== LEAKY MODEL RESULTS ===")
print(f"Precision@50: {precision_at_k(leaky_tree_score, y, 50)}")
print(export_text(leaky_tree, feature_names=leaky_features))

# ─────────────────────────────────────────────
# 3) DELETE THE LEAKY FEATURE
# ─────────────────────────────────────────────

# Drop the column that came from the label itself.
# This is the only score we trust.
honest_features_only = ["feat_impressions", "feat_clicks", "feat_avg_position", "feat_days_active", "feat_ctr"]

X_honest = feature_frame[honest_features_only].replace([np.inf, -np.inf], np.nan).fillna(0)

tree_honest = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_honest.fit(X_honest, y)

honest_score = tree_honest.predict_proba(X_honest)[:, 1]

print("\n=== HONEST MODEL (leaky feature REMOVED) ===")
print(f"Precision@50: {precision_at_k(honest_score, y, 50)}")
print(export_text(tree_honest, feature_names=honest_features_only))
print("\nThis is the score we keep. The 1.0 above was a mirage.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Grain Check ===
Rows violating the grain: 0
Empty DataFrame
Columns: [content_hash_id, client_hash_id, report_date, row_count]
Index: []
If the above returns zero rows then it will be confirmed that there are no duplicate values in the dataset
 One Row = One Page per Client per Day

=== Count and Span ===
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Availability Check ===
   total_rows  gsc_available_rows  ga4_available_rows
0     9841378           3611061.0            413966.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Feature Frame ===
(151981, 8)
            content_hash_id           client_hash_id  feat_impressions  \
0  content_ac8663da7484669a  client_62f4a7e64f5e0096              20.0   
1  content_39d7361b4945d504  client_62f4a7e64f5e0096              57.0   
2  content_d49a012dcb924e31  client_62f4a7e64f5e0096             246.0   
3  content_614baf2af4330bd7  client_62f4a7e64f5e0096             413.0   
4  content_225dc9235023be5f  client_62f4a7e64f5e0096             279.0   

   feat_clicks  feat_avg_position  feat_days_active  feat_ctr  declining_flag  
0          0.0           4.625000                 9  0.000000            True  
1          0.0           4.222711                15  0.000000            True  
2          0.0           4.520919                15  0.000000            True  
3          1.0           4.390322                15  0.002421            True  
4          1.0           9.961993                15  0.003584            True  

=== SELECTED FEATURE MODEL ===
Precisio

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This Slice of Data cannot tell you about the clients without active tracking in the month of march 2026. only 36.7% of rows have gsc_data_available = TRUE and only 4.2% have
ga4_data_available = TRUE (Query 3), meaning any label or feature built here reflects a
specific, non-random subset of clients, not the full warehouse population.

This Slice has no visibility beyond the 31 days, whether the page was declining before, keeps declining or recovers immediately after.

This slice also cannot reliably distinguish a genuine sustained decline from short-term noise.
The declining_flag is based on a first-half vs. second-half split within a single 31-day month

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.